In [1]:
import os
import torch
import json
from datasets import load_dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont

import torch
import torch.nn as nn
import torchvision.transforms as v2
from transformers import  ConditionalDetrImageProcessor, ConditionalDetrForObjectDetection
from transformers import ViTConfig, ViTImageProcessor, ViTModel

In [ ]:
device = torch.device('mps') if torch.backends.mps.is_available() else torch.device('cpu')
print(f'device: {device}')

In [ ]:
ckpt = "../object_detection/model_ckpt"
object_detection_image_processor = ConditionalDetrImageProcessor.from_pretrained(ckpt)
object_detector = ConditionalDetrForObjectDetection.from_pretrained(ckpt).to(device)
object_detector.eval()

In [4]:
def calculate_iou(box1, box2):
    x1, y1, x2, y2 = box1
    x3, y3, x4, y4 = box2

    xx1, yy1 = max(x1, x3), max(y1, y3)
    xx2, yy2 = min(x2, x4), min(y2, y4)

    intersection_area = max(0, xx2 - xx1) * max(0, yy2 - yy1)

    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x4 - x3) * (y4 - y3)

    iou = intersection_area / float(box1_area + box2_area - intersection_area)
    return iou

def non_max_suppression(items, iou_threshold=0.75):
    sorted_items = sorted(items, key=lambda x: x[0], reverse=True)
    
    keep = []
    while sorted_items:
        current = sorted_items.pop(0)
        keep.append(current)
        
        sorted_items = [
            item for item in sorted_items
            if calculate_iou(current[2], item[2]) < iou_threshold
        ]
    
    return keep

def detect_objects(image_path):
    image = Image.open(image_path).convert("RGB")

    with torch.no_grad():
        inputs = object_detection_image_processor(images=[image], return_tensors="pt")
        with torch.no_grad():
            outputs = object_detector(**inputs.to(device))
        target_sizes = torch.tensor([[image.size[1], image.size[0]]])
        results = object_detection_image_processor.post_process_object_detection(outputs, threshold=0.4, target_sizes=target_sizes)[0]

    items = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        box = [i.item() for i in box]
        items.append((score.item(), label.item(), box))
    
    # NMS 적용
    items = non_max_suppression(items)
        
    # draw = ImageDraw.Draw(image)
    # font = ImageFont.load_default(size=20)
    result = []
    for score, label, bbox in items:
        box = [round(i, 2) for i in bbox]
        result.append((image.crop(bbox), object_detector.config.id2label[label], score))
        # x, y, x2, y2 = tuple(bbox)
        # draw.rectangle((x, y, x2, y2), outline="green", width=2)
        # text_position = (bbox[0], bbox[1] - 25)  # bbox 위에 텍스트 위치
        # draw.text(text_position, f"{object_detector.config.id2label[label]} {score:.2f}", fill="yellow", font=font)

    # plt.figure(figsize=(12, 9))
    # plt.imshow(image)
    # plt.xticks([])
    # plt.yticks([])
    # plt.show()

    return result

In [5]:
labels = ['bag', 'bottom', 'dress', 'eyewear', 'hat', 'shoes', 'outer', 'top', 'watch']
id2label = {
    i: l for i, l in enumerate(labels)
}

In [ ]:
id2label

In [ ]:
ckpt = 'google/vit-base-patch16-224-in21k'
config = ViTConfig.from_pretrained(ckpt)
vit_image_processor = ViTImageProcessor.from_pretrained(ckpt)

transform = v2.Compose([
    v2.Resize((config.image_size, config.image_size)),
    v2.ToTensor(),
    v2.Normalize(mean=vit_image_processor.image_mean, std=vit_image_processor.image_std),
])

class Classifier(nn.Module):
    def __init__(self, num_labels):
        super(Classifier, self).__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.fc = nn.Linear(config.hidden_size, num_labels)
        
    def forward(self, x):
        logits = self.fc(self.vit(x).pooler_output)
        return logits

classifier = Classifier(num_labels=len(id2label)).to(device)
classifier.load_state_dict(torch.load('./classification_model/classifier.pt'))
classifier.eval()

In [ ]:
if not os.path.isdir('./onthelook_image_pairs'):
    os.makedirs('./onthelook_image_pairs')

already_post_ids = set()
label_dirs = [d for d in os.listdir('./onthelook_image_pairs') if not d.startswith('.')]
for label_dir in label_dirs:
    already_post_ids.update([f for f in os.listdir(f'./onthelook_image_pairs/{label_dir}') if not f.startswith('.')])

print(f'already post ids: {len(already_post_ids)}')

In [ ]:
onthelook_images_dir = f'../crawl/onthelook_images'
post_ids = [f for f in os.listdir(onthelook_images_dir) if not f.startswith('.') and f not in already_post_ids]
print(f'post_ids: {len(post_ids)}')

In [ ]:
for post_id in tqdm(post_ids[0:27000]):
    post_image_path = f'{onthelook_images_dir}/{post_id}/post.jpg'
    product_image_fnames = [f for f in os.listdir(f'{onthelook_images_dir}/{post_id}') if f.startswith('product')]
    if not product_image_fnames:
        continue

    try:
        result = detect_objects(post_image_path)
    except Exception:
        continue

    detected_labels = []
    shoes_count = 0
    detected_label_to_image = {}
    for cropped_image, label, score in result:
        if label == 'shoes':
            shoes_count += 1
        detected_labels.append(label)
        if label in detected_label_to_image:
            if score > detected_label_to_image[label][1]:
                detected_label_to_image[label] = ((cropped_image, score))
        else:
            detected_label_to_image[label] = ((cropped_image, score))

    for label, (cropped_image, score) in detected_label_to_image.items():
        detected_label_to_image[label] = cropped_image

    if shoes_count > 2:
        continue

    product_image_paths = [f'{onthelook_images_dir}/{post_id}/{product_image_fname}' for product_image_fname in product_image_fnames]
    tag_labels = []
    tag_label_to_image = {}
    for product_image_path in product_image_paths:
        try:
            image = Image.open(product_image_path).convert('RGB')
        except Exception:
            continue

        if not (image.width >= 200 and image.height >= 200):
            continue

        image_tensor = transform(image).to(device)
        with torch.no_grad():
            logits = classifier(image_tensor.unsqueeze(0)).squeeze()
        
        probs = torch.softmax(logits, dim=-1)
        indices = probs.argsort(dim=-1, descending=True).tolist()
        max_prob = probs[indices[0]].item()
        if max_prob < 0.8:
            continue

        label = id2label[indices[0]]
        if label in ('watch', 'eyewear'):
            continue

        tag_labels.append(label)
        tag_label_to_image[label] = image

    if not tag_labels:
        continue

    if not len(tag_labels) == len(set(tag_labels)):
        continue

    for label in tag_labels:
        anchor_image = detected_label_to_image.get(label)
        if not anchor_image:
            continue
        positive_image = tag_label_to_image[label]

        save_dir = f'./onthelook_image_pairs/{label}/{post_id}'
        if not os.path.isdir(save_dir):
            os.makedirs(save_dir)

        anchor_image_path = f'{save_dir}/anchor.jpg'
        positive_image_path = f'{save_dir}/positive.jpg'

        anchor_image.save(anchor_image_path)
        positive_image.save(positive_image_path)